# AdDhakhiraCorpusAI - Assistant Bibliographique Web App (Gemini)

## Utilisation
1. Clique sur le bouton **Play** de la cellule `Initialiser`.
2. Attends le message `Initialisation terminee`.
3. Clique sur le bouton **Play** de la cellule `Ouvrir l'interface`.
4. Ecris ta question dans la zone de chat puis clique sur **Envoyer**.

Tu n'as pas besoin de modifier le code.


## Etape 1 - Initialiser
Clique sur **Play**. Cette étape permet l'installation de AdDhakhira sur Google Drive et prépare la configuration.


In [ ]:
#@title Etape 1 - Initialiser
REPO_GIT_URL = "https://github.com/git-haddadz/AdDhakhiraCorpusAI.git" #@param {type:"string"}
REPO_DIR = "/content/AdDhakhiraCorpusAI" #@param {type:"string"}
DRIVE_DIR = "/content/drive/MyDrive/AdDhakhiraCorpusAI" #@param {type:"string"}
GEMINI_API_KEY = "" #@param {type:"string"}
GEMINI_MODEL = "gemini-2.5-flash" #@param {type:"string"}

import os
import sys
import subprocess
from pathlib import Path
from google.colab import drive

if not GEMINI_API_KEY.strip():
    raise ValueError('GEMINI_API_KEY est requis.')

drive.mount('/content/drive')
repo_path = Path(REPO_DIR)
if not (repo_path / 'main.py').exists():
    if repo_path.exists():
        subprocess.run(['rm', '-rf', str(repo_path)], check=True)
    subprocess.run(['git', 'clone', REPO_GIT_URL, str(repo_path)], check=True)
os.chdir(repo_path)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'requests', 'transformers', 'rank-bm25', 'sentence-transformers', 'faiss-cpu', 'numpy', 'scipy', 'scikit-learn'], check=True)

project_dir = Path(DRIVE_DIR)
output_dir = project_dir / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

config_text = f'''MODEL_EXTRACTOR_PATH = "{GEMINI_MODEL}"
MODEL_REASONER_PATH = "{GEMINI_MODEL}"
LLM_BACKEND = "gemini_api"
GEMINI_API_KEY = "{GEMINI_API_KEY}"
JSON_INPUT_PATH = "{repo_path / 'database'}"
TOP_K_CHUNKS = 8
TOP_K_PAGES = 5
MAX_MODEL_LEN_EXTRACTOR = 4096
MIN_MODEL_LEN_REASONER = 8192
NUM_GPUS_EXTRACTOR = 1
NUM_GPUS_REASONER = 1
EMBEDDING_MODEL = None
TRANSLATE_TOP_CHUNKS_TO_FRENCH = False
TRANSLATION_MAX_TOKENS = 4096
AUTO_TRANSLATE_QUESTION_TO_ARABIC = False
REASONER_TEMPERATURE = 0.2
REASONER_TOP_P = 0.9
REASONER_OUTPUT_MAX_TOKENS = 1200
REASONER_CONTEXT_SAFETY_TOKENS = 8000
'''
(repo_path / 'src' / 'config.py').write_text(config_text, encoding='utf-8')
print('Initialisation terminee')
print('Repo:', repo_path)
print('Outputs:', output_dir)


## Etape 2 - Ouvrir l'interface
Clique sur **Play**. Une interface de chat apparaît juste en dessous.


In [ ]:
#@title Etape 2 - Ouvrir l'interface
import os
import time
import threading
from datetime import datetime
from pathlib import Path
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import output as colab_output
from src.pipeline import build_final_report
from src.reporting import write_output_with_timing

drive_dir = Path('/content/drive/MyDrive/AdDhakhiraCorpusAI')
session_dir = drive_dir / 'outputs' / f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
session_dir.mkdir(parents=True, exist_ok=True)

if not globals().get('_adhakhira_server_started', False):
    os.chdir(drive_dir)
    _adhakhira_server = ThreadingHTTPServer(('0.0.0.0', 8000), SimpleHTTPRequestHandler)
    _adhakhira_thread = threading.Thread(target=_adhakhira_server.serve_forever, daemon=True)
    _adhakhira_thread.start()
    globals()['_adhakhira_server_started'] = True

base_url = colab_output.eval_js('google.colab.kernel.proxyPort(8000)')
title = widgets.HTML('<h3>Assistant de recherche Malikite</h3><p>Pose ta question.</p>')
box = widgets.Textarea(value='', placeholder='Ecris ta question ici', description='Question:', layout=widgets.Layout(width='100%', height='100px'))
btn = widgets.Button(description='Envoyer', button_style='primary')
status = widgets.HTML('')
chat = widgets.Output()
counter = {'n': 0}

def on_ask(_):
    question = box.value.strip()
    if not question:
        status.value = '<b>Saisis une question.</b>'
        return
    btn.disabled = True
    status.value = '<i>Traitement en cours...</i>'
    counter['n'] += 1
    out_path = session_dir / f'output_{counter["n"]}.html'
    t0 = time.time()
    try:
        report = build_final_report(question=question, translate_to_french=False, diagnostic_coherence=True)
        elapsed = time.time() - t0
        write_output_with_timing(report, elapsed_seconds=elapsed, output_path=str(out_path))
        rel = out_path.relative_to(drive_dir)
        url = f"{base_url}/{rel.as_posix()}"
        with chat:
            display(HTML(f"<div style='border:1px solid #ddd;border-radius:10px;padding:12px;margin:10px 0;background:#fafafa'><p><b>Question:</b> {question}</p><p><a href='{url}' target='_blank'>Ouvrir la reponse HTML</a></p><p><small>{out_path}</small></p></div>"))
        status.value = '<b>Termine</b>'
        box.value = ''
    except Exception as e:
        status.value = f'<b>Erreur:</b> {e}'
    finally:
        btn.disabled = False

btn.on_click(on_ask)
display(widgets.VBox([title, box, btn, status, chat]))
print('Dossier de sortie:', session_dir)
